# Laboratorio 04 — Codificación de Canal

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ollerenac/wireless-communication-systems/blob/main/docs/sessions/04-channel-coding/lab.ipynb)

**Objetivos:**
- Visualizar la capacidad de Shannon y el límite de $E_b/N_0$
- Verificar el síndromo de un código de bloque en $\mathrm{GF}(2)$
- Simular belief propagation en un código LDPC pequeño
- Calcular los parámetros de Bhattacharyya y comparar canales polares
- Generar curvas waterfall para LDPC y Polar

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.special import erfc
from itertools import product
import os

plt.rcParams.update({
    'figure.dpi': 120,
    'font.size': 11,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.figsize': (10, 5),
})
os.makedirs('figures', exist_ok=True)

def Q(x):
    return 0.5 * erfc(x / np.sqrt(2))

/home/researcher/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


---
## Ejercicio 1 — Capacidad de Shannon y límite $E_b/N_0$

La capacidad del canal AWGN de banda $B$ es
$$C = B \log_2\!\left(1 + \frac{S}{N_0 B}\right) \quad [\text{bit/s}]$$

Expresada en función de la eficiencia espectral $\eta = C/B$:
$$\frac{E_b}{N_0} = \frac{2^\eta - 1}{\eta}$$

**Tareas:**
1. Grafica $\eta$ vs $E_b/N_0$ (dB) para $\eta \in [0.1, 6]$ bit/s/Hz.
2. Marca el límite de Shannon ($\eta \to 0$, $E_b/N_0 \to \ln 2 \approx -1{,}59$ dB).
3. Marca el punto de operación de 64-QAM 3/4 ($\eta = 4{,}5$ bit/s/Hz, $E_b/N_0 \approx 10$ dB).

In [ ]:
eta = np.linspace(0.05, 6, 500)
EbN0_lin = (2**eta - 1) / eta
EbN0_dB = 10*np.log10(EbN0_lin)
shannon_limit_dB = 10*np.log10(np.log(2))

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(EbN0_dB, eta, 'b-', lw=2.5, label='Límite de Shannon $C = \\log_2(1+SNR)$')
ax.fill_betweenx(eta, -3, EbN0_dB, alpha=0.12, color='salmon', label='Región no alcanzable')
ax.axvline(shannon_limit_dB, color='red', ls='--', lw=1.5,
           label=f'Límite absoluto {shannon_limit_dB:.2f} dB')

MCS = [
    ('BPSK $r=1/2$',   0.5, 3.0,  'steelblue',      +0.2),
    ('QPSK $r=1/2$',   1.0, 6.0,  'mediumseagreen',  +0.2),
    ('QPSK $r=3/4$',   1.5, 9.0,  'mediumseagreen',  -0.3),
    ('16-QAM $r=1/2$', 2.0, 13.0, 'darkorange',      +0.2),
    ('64-QAM $r=3/4$', 4.5, 22.0, 'firebrick',       +0.2),
]

for label, eta_op, ebno_op, color, dy in MCS:
    ax.scatter([ebno_op], [eta_op], s=70, color=color, zorder=5)
    ax.annotate(label, xy=(ebno_op, eta_op),
                xytext=(ebno_op + 0.5, eta_op + dy),
                fontsize=8, color=color,
                arrowprops=dict(arrowstyle='->', color=color, lw=0.8))
    ebno_shannon = 10*np.log10((2**eta_op - 1)/eta_op)
    gap = ebno_op - ebno_shannon
    ax.annotate('', xy=(ebno_shannon, eta_op), xytext=(ebno_op, eta_op),
                arrowprops=dict(arrowstyle='<->', color='gray', lw=1.0))
    ax.text((ebno_op + ebno_shannon)/2, eta_op + 0.12,
            f'{gap:.1f} dB', ha='center', va='bottom', fontsize=7.5, color='gray')

ax.set_xlabel('$E_b/N_0$ (dB)')
ax.set_ylabel('Eficiencia espectral $\\eta$ (bit/s/Hz)')
ax.set_title('Región de Shannon — Capacidad del canal AWGN y puntos de operación 5G NR')
ax.set_xlim(-3, 25)
ax.set_ylim(0, 6.2)
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.savefig('figures/shannon-capacity.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Límite absoluto Eb/N0: {shannon_limit_dB:.2f} dB')
for label, eta_op, ebno_op, color, dy in MCS:
    ebno_shannon = 10*np.log10((2**eta_op - 1)/eta_op)
    print(f'{label}: gap = {ebno_op - ebno_shannon:.1f} dB')

---
## Ejercicio 2 — Código de bloque: síndrome y detección de errores

El código (7,4) de Hamming tiene la matriz de paridad:
$$H = \begin{pmatrix}
1 & 1 & 0 & 1 & 1 & 0 & 0 \\
1 & 0 & 1 & 1 & 0 & 1 & 0 \\
0 & 1 & 1 & 1 & 0 & 0 & 1
\end{pmatrix}$$

Un vector recibido válido cumple $H\mathbf{r}^\top = \mathbf{0}$ en $\mathrm{GF}(2)$.

**Tareas:**
1. Verifica que la palabra código $\mathbf{c} = [1,0,1,1,0,1,0]$ es válida.
2. Introduce un error en el bit 3 (índice 2) y calcula el síndrome.
3. Demuestra que el síndrome identifica la columna de $H$ que corresponde al bit erróneo.

In [3]:
# Matriz de paridad del código Hamming (7,4)
H = np.array([
    [1, 1, 0, 1, 1, 0, 0],
    [1, 0, 1, 1, 0, 1, 0],
    [0, 1, 1, 1, 0, 0, 1],
], dtype=int)

c = np.array([1, 0, 1, 1, 0, 1, 0], dtype=int)  # palabra código válida

def syndrome(H, r):
    """Síndrome s = H·r^T mod 2"""
    return H @ r % 2

# 1. Verificar palabra válida
s_valid = syndrome(H, c)
print(f'Síndrome de c (sin error): {s_valid}  →  {"válida" if np.all(s_valid == 0) else "ERROR"}')

# 2. Introducir error en bit 2 (índice 2)
r = c.copy()
error_pos = 2
r[error_pos] ^= 1
print(f'Palabra recibida con error en bit {error_pos}: {r}')

s_err = syndrome(H, r)
print(f'Síndrome: {s_err}')

# 3. Identificar columna de H igual al síndrome
for j in range(H.shape[1]):
    if np.array_equal(H[:, j], s_err):
        print(f'El síndrome coincide con la columna {j} de H → error en bit {j}')
        break

# Corregir
r_corr = r.copy()
r_corr[j] ^= 1
print(f'Palabra corregida: {r_corr}  →  {"igual a c" if np.array_equal(r_corr, c) else "diferente"}')

Síndrome de c (sin error): [0 0 0]  →  válida
Palabra recibida con error en bit 2: [1 0 0 1 0 1 0]
Síndrome: [0 1 1]
El síndrome coincide con la columna 2 de H → error en bit 2
Palabra corregida: [1 0 1 1 0 1 0]  →  igual a c


---
## Ejercicio 3 — LDPC: belief propagation simplificado

Usaremos el código LDPC (8, 4) con la siguiente matriz de paridad $H$ (tamaño $4 \times 8$):

| Nodo de chequeo | Bits conectados |
|---|---|
| $c_0$ | $v_0, v_1, v_2, v_3$ |
| $c_1$ | $v_1, v_2, v_4, v_5$ |
| $c_2$ | $v_2, v_3, v_5, v_6$ |
| $c_3$ | $v_0, v_3, v_6, v_7$ |

Implementaremos el **algoritmo sum-product** (propagación de creencia) para BSC con probabilidad de error $p$.

**Tareas:**
1. Construye $H$ y genera una palabra código válida.
2. Transmite sobre un BSC con $p = 0{,}1$ e introduce errores.
3. Ejecuta 20 iteraciones de BP y verifica si se recupera la palabra original.

In [ ]:
# Código LDPC (8,4)
H_ldpc = np.array([
    [1, 1, 1, 1, 0, 0, 0, 0],
    [0, 1, 1, 0, 1, 1, 0, 0],
    [0, 0, 1, 1, 0, 1, 1, 0],
    [1, 0, 0, 1, 0, 0, 1, 1],
], dtype=int)

n, k = 8, 4  # longitud y dimensión
rng = np.random.default_rng(42)

def gf2_row_reduce(M):
    """Reducción por filas en GF(2). Devuelve la forma escalonada."""
    M = M.copy() % 2
    rows, cols = M.shape
    pivot_row = 0
    for col in range(cols):
        if pivot_row >= rows:
            break
        pivot = np.where(M[pivot_row:, col] == 1)[0]
        if len(pivot) == 0:
            continue
        pivot += pivot_row
        M[[pivot_row, pivot[0]]] = M[[pivot[0], pivot_row]]
        for r in range(rows):
            if r != pivot_row and M[r, col] == 1:
                M[r] ^= M[pivot_row]
        pivot_row += 1
    return M % 2

# Encontrar una palabra código: resolver H·c^T = 0 en GF(2)
# Palabra código simple: todos ceros
c_ldpc = np.zeros(n, dtype=int)
assert np.all(H_ldpc @ c_ldpc % 2 == 0), 'c no es palabra código'

# Otra palabra código (suma de filas generadoras)
# G se obtiene del espacio nulo de H en GF(2)
# Para simplificar, usamos una palabra no trivial verificada a mano:
c_ldpc = np.array([0, 1, 0, 1, 0, 1, 0, 1], dtype=int)
print(f'Síndrome de c_ldpc: {H_ldpc @ c_ldpc % 2}  →  {"válida" if np.all(H_ldpc @ c_ldpc % 2 == 0) else "inválida"}')


def bp_bsc(H, r, p, n_iter=20):
    """
    Belief Propagation (sum-product) para BSC con prob. de error p.
    r: vector recibido (bits 0/1)
    Devuelve: decisión dura tras n_iter iteraciones
    """
    m, n = H.shape
    # LLR del canal: L_ch[i] = log(P(r=0|x=0)/P(r=1|x=0))
    #   Si r_i = 0: L = log((1-p)/p)   (favor de 0)
    #   Si r_i = 1: L = log(p/(1-p))   (favor de 1 → negativo)
    L_ch = np.where(r == 0, np.log((1-p)/p), np.log(p/(1-p)))

    # Mensajes v→c (LLR)
    L_vc = np.zeros((m, n))
    for i in range(m):
        for j in range(n):
            if H[i, j]:
                L_vc[i, j] = L_ch[j]

    for it in range(n_iter):
        # Paso c→v: mensaje del nodo de chequeo c_i al nodo variable v_j
        L_cv = np.zeros((m, n))
        for i in range(m):
            nbrs = np.where(H[i, :] == 1)[0]  # vecinos de c_i
            for j in nbrs:
                other = [jj for jj in nbrs if jj != j]
                if len(other) == 0:
                    L_cv[i, j] = 0.0
                    continue
                # Producto de tanh (sum-product exacto)
                prod_tanh = np.prod(np.tanh(L_vc[i, other] / 2))
                # Clamp para evitar atanh(±1)
                prod_tanh = np.clip(prod_tanh, -1 + 1e-10, 1 - 1e-10)
                L_cv[i, j] = 2 * np.arctanh(prod_tanh)

        # Paso v→c: mensaje actualizado
        for j in range(n):
            nbrs_c = np.where(H[:, j] == 1)[0]  # nodos de chequeo que inciden en v_j
            total_llr = L_ch[j] + np.sum(L_cv[nbrs_c, j])
            for i in nbrs_c:
                L_vc[i, j] = total_llr - L_cv[i, j]

        # LLR marginal
        L_total = np.array([
            L_ch[j] + np.sum(L_cv[np.where(H[:, j] == 1)[0], j])
            for j in range(n)
        ])
        c_hat = (L_total < 0).astype(int)  # decide 1 si LLR < 0

        # Verificar síndrome
        if np.all(H @ c_hat % 2 == 0):
            return c_hat, it + 1

    return c_hat, n_iter


# Canal BSC con p=0.1
p_bsc = 0.1
errors = (rng.random(n) < p_bsc).astype(int)
r_ldpc = (c_ldpc + errors) % 2
print(f'\nPalabra enviada:    {c_ldpc}')
print(f'Errores en posic.:  {np.where(errors)[0].tolist()}')
print(f'Palabra recibida:   {r_ldpc}')

c_hat, iters = bp_bsc(H_ldpc, r_ldpc, p_bsc, n_iter=20)
print(f'\nDecisión BP ({iters} iters): {c_hat}')
print(f'Síndrome: {H_ldpc @ c_hat % 2}')
print(f'Recuperación: {"CORRECTA" if np.array_equal(c_hat, c_ldpc) else "INCORRECTA"}')

# ── Grafo de Tanner — FIG-04 ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.set_aspect('equal')
ax.axis('off')

n_v, n_c = H_ldpc.shape[1], H_ldpc.shape[0]   # 8 variable, 4 check nodes

# Posiciones: nodos de variable arriba, nodos de verificación abajo
# c_x centrado sobre el mismo span que v_x para layout bipartito simétrico
v_x = np.linspace(0.5, n_v - 0.5, n_v)
c_x = np.linspace(0.5, n_v - 0.5, n_c)
v_y, c_y = 1.5, 0.0

# Aristas
for i in range(n_c):
    for j in range(n_v):
        if H_ldpc[i, j]:
            ax.plot([v_x[j], c_x[i]], [v_y, c_y], 'k-', lw=0.8, alpha=0.5, zorder=1)

# Nodos de variable (círculos)
for j, x in enumerate(v_x):
    circle = plt.Circle((x, v_y), 0.28, color='steelblue', zorder=3)
    ax.add_patch(circle)
    ax.text(x, v_y, f'$v_{j}$', ha='center', va='center', fontsize=9,
            color='white', fontweight='bold', zorder=4)

# Nodos de verificación (cuadrados)
for i, x in enumerate(c_x):
    sq = plt.Rectangle((x - 0.28, c_y - 0.28), 0.56, 0.56,
                        color='darkorange', zorder=3)
    ax.add_patch(sq)
    ax.text(x, c_y, f'$c_{i}$', ha='center', va='center', fontsize=9,
            color='white', fontweight='bold', zorder=4)

ax.set_xlim(-0.2, n_v + 0.2)
ax.set_ylim(-0.7, 2.2)
ax.set_title('Grafo de Tanner — código LDPC (8, 4)', fontsize=12)
plt.tight_layout()
plt.savefig('figures/tanner-graph.png', dpi=150, bbox_inches='tight')
plt.show()

In [5]:
# ── Ejercicio 3 (cont.) — LDPC realista: código Gallager n=240 ───────────
# Código de longitud n=240, tasa ~1/2 y ~3/4, grado de variable d_v=3
# Construido con la matriz de Gallager (apilamiento de submatrices permutadas)

def gallager_ldpc(n, d_v, d_c, seed=2024):
    """
    Matriz de paridad LDPC regular de Gallager.
    Requiere: n * d_v % d_c == 0  Y  (n * d_v // d_c) % d_v == 0
    Devuelve H de forma (n*d_v//d_c, n)
    """
    rng = np.random.default_rng(seed)
    m = n * d_v // d_c
    sub_m = m // d_v          # filas por submatriz
    # Primera submatriz: d_c columnas consecutivas por fila
    H0 = np.zeros((sub_m, n), dtype=int)
    for i in range(sub_m):
        for j in range(i * d_c, (i+1) * d_c):
            H0[i, j % n] = 1
    # Apilar d_v submatrices, cada una es una permutación de columnas de H0
    rows = [H0]
    for _ in range(d_v - 1):
        perm = rng.permutation(n)
        rows.append(H0[:, perm])
    return np.vstack(rows)

def gf2_rank(M):
    """Rango de M en GF(2) por eliminación de Gauss."""
    M = M.copy() % 2
    rows, cols = M.shape
    pivot_row = 0
    for col in range(cols):
        if pivot_row >= rows:
            break
        pivot = np.where(M[pivot_row:, col] == 1)[0]
        if len(pivot) == 0:
            continue
        pivot += pivot_row
        M[[pivot_row, pivot[0]]] = M[[pivot[0], pivot_row]]
        for r in range(rows):
            if r != pivot_row and M[r, col] == 1:
                M[r] ^= M[pivot_row]
        pivot_row += 1
    return pivot_row

# Construir los dos códigos
H12 = gallager_ldpc(240, d_v=3, d_c=6,  seed=2024)   # tasa ~1/2
H34 = gallager_ldpc(240, d_v=3, d_c=12, seed=2024)   # tasa ~3/4

k12 = 240 - gf2_rank(H12)   # ≈ 122
k34 = 240 - gf2_rank(H34)   # ≈ 182
Rc12, Rc34 = k12 / 240, k34 / 240
print(f"H12: {H12.shape}, k={k12}, tasa={Rc12:.3f}")
print(f"H34: {H34.shape}, k={k34}, tasa={Rc34:.3f}")

# Tablas de vecinos (calcular una vez)
var_nbrs12 = [np.where(H12[:, j] == 1)[0] for j in range(240)]
chk_nbrs12 = [np.where(H12[i, :] == 1)[0] for i in range(H12.shape[0])]
var_nbrs34 = [np.where(H34[:, j] == 1)[0] for j in range(240)]
chk_nbrs34 = [np.where(H34[i, :] == 1)[0] for i in range(H34.shape[0])]

def bp_awgn(H, llr_ch, var_nbrs, chk_nbrs, max_iter=30):
    """
    Decodificador BP sum-product para BPSK AWGN.
    H:         matriz de paridad (m x n)
    llr_ch:    LLRs del canal, shape (n,) = 2*y/sigma^2
    var_nbrs:  lista longitud n; var_nbrs[j] = indices de check nodes adyacentes a v_j
    chk_nbrs:  lista longitud m; chk_nbrs[i] = indices de variable nodes adyacentes a c_i
    Devuelve:  (c_hat, n_iters_used)
    """
    m, n = H.shape
    # Inicializar mensajes v->c con LLR del canal
    L_vc = np.zeros((m, n))
    for i in range(m):
        L_vc[i, chk_nbrs[i]] = llr_ch[chk_nbrs[i]]
    L_cv = np.zeros((m, n))

    for it in range(max_iter):
        # Actualizacion c->v (regla tanh / sum-product)
        for i in range(m):
            nbrs = chk_nbrs[i]
            tanh_vals = np.tanh(np.clip(L_vc[i, nbrs] / 2, -20, 20))
            prod_all = np.prod(tanh_vals)
            for idx, j in enumerate(nbrs):
                # excluir v_j del producto
                t_excl = prod_all / (tanh_vals[idx] + 1e-300)
                t_excl = np.clip(t_excl, -1 + 1e-12, 1 - 1e-12)
                L_cv[i, j] = 2 * np.arctanh(t_excl)

        # Actualizacion v->c
        for j in range(n):
            nbrs = var_nbrs[j]
            total = llr_ch[j] + np.sum(L_cv[nbrs, j])
            L_vc[nbrs, j] = total - L_cv[nbrs, j]

        # Decision dura y verificacion de sindrome
        L_total = np.array([
            llr_ch[j] + np.sum(L_cv[var_nbrs[j], j]) for j in range(n)
        ])
        c_hat = (L_total < 0).astype(int)
        if np.all(H @ c_hat % 2 == 0):
            return c_hat, it + 1   # convergio

    return c_hat, max_iter   # no convergio

def bp_awgn_track(H, llr_ch, var_nbrs, chk_nbrs, max_iter=30, track_iters=(1, 3, 10)):
    """Como bp_awgn pero registra L_total en las iteraciones especificadas."""
    m, n = H.shape
    L_vc = np.zeros((m, n))
    for i in range(m):
        L_vc[i, chk_nbrs[i]] = llr_ch[chk_nbrs[i]]
    L_cv = np.zeros((m, n))
    history = {}

    for it in range(max_iter):
        for i in range(m):
            nbrs = chk_nbrs[i]
            tanh_vals = np.tanh(np.clip(L_vc[i, nbrs] / 2, -20, 20))
            prod_all = np.prod(tanh_vals)
            for idx, j in enumerate(nbrs):
                t_excl = prod_all / (tanh_vals[idx] + 1e-300)
                t_excl = np.clip(t_excl, -1 + 1e-12, 1 - 1e-12)
                L_cv[i, j] = 2 * np.arctanh(t_excl)
        for j in range(n):
            nbrs = var_nbrs[j]
            total = llr_ch[j] + np.sum(L_cv[nbrs, j])
            L_vc[nbrs, j] = total - L_cv[nbrs, j]
        L_total = np.array([llr_ch[j] + np.sum(L_cv[var_nbrs[j], j]) for j in range(n)])
        if (it + 1) in track_iters:
            history[it + 1] = L_total.copy()
        c_hat = (L_total < 0).astype(int)
        if np.all(H @ c_hat % 2 == 0):
            for k in track_iters:
                if k not in history:
                    history[k] = L_total.copy()
            return c_hat, it + 1, history

    for k in track_iters:
        if k not in history:
            history[k] = L_total.copy()
    return c_hat, max_iter, history

# Verificacion rapida: convergencia a 5 dB
rng_test = np.random.default_rng(0)
EbN0_test = 5.0
sigma2_test = 1.0 / (2 * Rc12 * 10**(EbN0_test/10))
y_test = np.ones(240) + np.sqrt(sigma2_test) * rng_test.standard_normal(240)
_, iters_test = bp_awgn(H12, 2*y_test/sigma2_test, var_nbrs12, chk_nbrs12)
print(f"Verificacion BP a {EbN0_test} dB: convergio en {iters_test} iteraciones (esperado <15)")
assert iters_test < 15, f"BP tardo {iters_test} iter — revisar sigma2"

H12: (120, 240), k=122, tasa=0.508
H34: (60, 240), k=182, tasa=0.758
Verificacion BP a 5.0 dB: convergio en 2 iteraciones (esperado <15)


In [6]:
# ── bp-messages.png — FIG-05: evolucion de LLR marginales ────────────────
rng_fig = np.random.default_rng(137)   # semilla fija para figura reproducible
EbN0_track_dB = 2.5
EbN0_track_lin = 10**(EbN0_track_dB / 10)
sigma2_track = 1.0 / (2 * Rc12 * EbN0_track_lin)
sigma_track = np.sqrt(sigma2_track)

c_track = np.zeros(240, dtype=int)
bpsk_track = np.ones(240)
y_track = bpsk_track + sigma_track * rng_fig.standard_normal(240)
llr_track = 2 * y_track / sigma2_track

_, _, history = bp_awgn_track(H12, llr_track, var_nbrs12, chk_nbrs12,
                               max_iter=30, track_iters=(1, 3, 10))

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)  # excepcion al estandar (10,5): ancho extra necesario para 3 paneles horizontales
bins = np.linspace(-25, 25, 50)
for ax, it_num in zip(axes, [1, 3, 10]):
    L = history[it_num]
    ax.hist(L, bins=bins, color='steelblue', edgecolor='white', linewidth=0.3)
    ax.axvline(0, color='darkorange', ls='--', lw=1.5, label='Umbral de decision')
    ax.set_title(f'Iteracion {it_num}', fontsize=11)
    ax.set_xlabel('LLR marginal $\\lambda_v^{(\\mathrm{total})}$')
    n_errors = np.sum((L < 0) != c_track)
    ax.text(0.97, 0.97, f'{n_errors} errores', transform=ax.transAxes,
            ha='right', va='top', fontsize=9, color='firebrick')
axes[0].set_ylabel('Numero de bits')
axes[1].set_title(f'Iteracion 3  (Eb/N0 = {EbN0_track_dB} dB)')
fig.suptitle('Convergencia de BP: evolucion de LLR marginales — codigo LDPC n=240, r=1/2',
             fontsize=11, y=1.01)
plt.tight_layout()
plt.savefig('figures/bp-messages.png', dpi=150, bbox_inches='tight')
plt.show()

/tmp/claude-1000/ipykernel_867002/2586508634.py:33: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# ── ldpc-ber-waterfall.png — FIG-08: curvas BER Monte Carlo ──────────────
# NOTA: esta celda tarda ~40 segundos en total (n_blocks=200, n=240)

rng_mc = np.random.default_rng(42)

def run_mc_ber(H, var_nbrs, chk_nbrs, Rc, EbN0_dB_arr, n_blocks=200, max_iter=30):
    n = H.shape[1]
    floor = 1.0 / (n_blocks * n)
    c_test = np.zeros(n, dtype=int)
    bpsk = np.ones(n)
    BER = []
    for EbN0_dB in EbN0_dB_arr:
        EbN0_lin = 10**(EbN0_dB / 10)
        sigma2 = 1.0 / (2 * Rc * EbN0_lin)
        sigma = np.sqrt(sigma2)
        bit_errors = 0
        for _ in range(n_blocks):
            y = bpsk + sigma * rng_mc.standard_normal(n)
            c_hat, _ = bp_awgn(H, 2*y/sigma2, var_nbrs, chk_nbrs, max_iter)
            bit_errors += np.sum(c_hat != c_test)
        BER.append(max(bit_errors / (n_blocks * n), floor))
    return np.array(BER)

EbN0_12 = np.arange(0.0, 5.5, 0.5)
EbN0_34 = np.arange(2.5, 8.0, 0.5)

print("Simulando curva BER, tasa 1/2 (puede tardar ~25 s)...")
BER12 = run_mc_ber(H12, var_nbrs12, chk_nbrs12, Rc12, EbN0_12, n_blocks=200)
print("Simulando curva BER, tasa 3/4 (puede tardar ~20 s)...")
BER34 = run_mc_ber(H34, var_nbrs34, chk_nbrs34, Rc34, EbN0_34, n_blocks=200)

# Verificar waterfall visible
ratio = BER12[0] / BER12[-1]
print(f"BER tasa 1/2: {BER12[0]:.2e} -> {BER12[-1]:.2e}, ratio={ratio:.0f}x")
assert ratio > 1000, f"Waterfall no visible: ratio={ratio:.0f} (esperado >1000)"

# ── Figura ────────────────────────────────────────────────────────────────
EbN0_plot = np.linspace(-1, 9, 300)
BER_bpsk = 0.5 * erfc(np.sqrt(10**(EbN0_plot/10)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(EbN0_plot, BER_bpsk, 'k-', lw=2, label='BPSK sin codigo')
ax.semilogy(EbN0_12, BER12, 'o-', color='steelblue', lw=2, ms=5,
            label=f'LDPC $r_c\\approx{Rc12:.2f}$ (n=240, Monte Carlo)')
ax.semilogy(EbN0_34, BER34, 's--', color='darkorange', lw=2, ms=5,
            label=f'LDPC $r_c\\approx{Rc34:.2f}$ (n=240, Monte Carlo)')

# Limites de Shannon por tasa
for r, color in [(Rc12, 'steelblue'), (Rc34, 'darkorange')]:
    ebn0_sh = 10*np.log10((2**r - 1)/r)
    ax.axvline(ebn0_sh, color=color, ls=':', lw=0.9, alpha=0.6)
    ax.text(ebn0_sh + 0.05, 2e-1, f'$C(r={r:.2f})$', fontsize=8,
            color=color, rotation=90, va='top')

ax.set_xlabel('$E_b/N_0$ (dB)')
ax.set_ylabel('BER')
ax.set_title('Curvas waterfall LDPC — Monte Carlo (n=240, BP sum-product)')
ax.set_xlim(-1, 9)
ax.set_ylim(1e-5, 1.1)
ax.legend(fontsize=9, loc='lower left')
plt.tight_layout()
plt.savefig('figures/ldpc-ber-waterfall.png', dpi=150, bbox_inches='tight')
plt.show()

Simulando curva BER, tasa 1/2 (puede tardar ~25 s)...


Simulando curva BER, tasa 3/4 (puede tardar ~20 s)...


BER tasa 1/2: 1.24e-01 -> 2.08e-05, ratio=5969x


/tmp/claude-1000/ipykernel_867002/1382854520.py:63: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Ejercicio 4 — Polar codes: árbol de Bhattacharyya

Para un canal binario simétrico (BEC) de parámetro de borrado $\varepsilon$, la transformación Arıkan define:
$$Z(W^-) = 2Z(W) - Z(W)^2 \qquad Z(W^+) = Z(W)^2$$

donde $Z(W) \in [0,1]$ es el parámetro de Bhattacharyya (0 = canal perfecto, 1 = canal completamente borrado).

**Tareas:**
1. Calcula el árbol completo de $N = 8$ canales polares a partir de $Z_0 = \varepsilon = 0{,}5$.
2. Identifica los canales «buenos» (frozen = 0) y los «de información».
3. Estima la probabilidad de bloque de error del código Polar (8, 4) resultante.

In [8]:
def bhattacharyya_tree(Z0, N):
    """
    Calcula los parámetros de Bhattacharyya para N canales polares
    partiendo de Z0 (parámetro del canal base).
    Devuelve lista de longitud N con Z[i] para canal i.
    """
    n_stages = int(np.log2(N))
    # Comenzar con [Z0]
    channels = [Z0]
    for stage in range(n_stages):
        new_channels = []
        for z in channels:
            z_minus = 2*z - z**2   # canal malo (peor)
            z_plus  = z**2          # canal bueno (mejor)
            new_channels.extend([z_minus, z_plus])
        channels = new_channels
    return np.array(channels)


N_polar = 8
eps = 0.5   # BEC con epsilon=0.5

Z_channels = bhattacharyya_tree(eps, N_polar)
print(f'Parámetros de Bhattacharyya para N={N_polar}, ε={eps}:')
for i, z in enumerate(Z_channels):
    print(f'  Canal {i:2d}: Z = {z:.4f}  →  {"INFORMACIÓN" if z < 0.5 else "frozen     "}')

# Seleccionar k=4 canales con menor Z (mejores)
k_polar = 4
info_indices = np.argsort(Z_channels)[:k_polar]
frozen_indices = np.argsort(Z_channels)[k_polar:]
print(f'\nCanales de información (k={k_polar}): {sorted(info_indices)}')
print(f'Canales frozen:                      {sorted(frozen_indices)}')

# Cota de probabilidad de error de bloque (unión)
P_ub = np.sum(Z_channels[info_indices])
print(f'\nCota superior P_b ≤ {P_ub:.4f}')

# Visualización
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['steelblue' if i in info_indices else 'salmon' for i in range(N_polar)]
bars = ax.bar(range(N_polar), Z_channels, color=colors, edgecolor='k', lw=0.5)
ax.axhline(0.5, color='gray', ls='--', lw=1, label='Umbral de selección')
ax.set_xticks(range(N_polar))
ax.set_xlabel('Índice del canal polar $i$')
ax.set_ylabel('$Z(W_N^{(i)})$')
ax.set_title(f'Árbol de Bhattacharyya: N={N_polar}, ε={eps}')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='steelblue', label='Canal de información'),
                   Patch(facecolor='salmon',    label='Canal frozen')]
ax.legend(handles=legend_elements, fontsize=9)
plt.tight_layout()
plt.show()

Parámetros de Bhattacharyya para N=8, ε=0.5:
  Canal  0: Z = 0.9961  →  frozen     
  Canal  1: Z = 0.8789  →  frozen     
  Canal  2: Z = 0.8086  →  frozen     
  Canal  3: Z = 0.3164  →  INFORMACIÓN
  Canal  4: Z = 0.6836  →  frozen     
  Canal  5: Z = 0.1914  →  INFORMACIÓN
  Canal  6: Z = 0.1211  →  INFORMACIÓN
  Canal  7: Z = 0.0039  →  INFORMACIÓN

Canales de información (k=4): [3, 5, 6, 7]
Canales frozen:                      [0, 1, 2, 4]

Cota superior P_b ≤ 0.6328


/tmp/claude-1000/ipykernel_867002/1297815149.py:53: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Ejercicio 5 — Decodificador SC para código Polar (N=4)

El decodificador Successive Cancellation (SC) recorre el árbol factor de la transformación Arıkan recursivamente. Para $N = 4$ podemos implementarlo explícitamente.

Las funciones de actualización de LLR son:
$$f(a, b) = 2\,\mathrm{arctanh}\!\left(\tanh\frac{a}{2}\cdot\tanh\frac{b}{2}\right) \quad (\text{nodo} \ {-})$$
$$g(a, b, u) = b + (1 - 2u)\,a \quad (\text{nodo} \ {+})$$

**Tareas:**
1. Implementa el decodificador SC para $N = 4$ con bits frozen $u_0 = u_1 = 0$ (canales 0 y 1 frozen).
2. Transmite la palabra código por un canal AWGN con SNR = 5 dB y decodifica.
3. Compara con decisión dura directa.

In [ ]:
def f_func(a, b):
    """Actualización de LLR para el nodo '-' (combinación XOR) — Ec. (9)"""
    t = np.tanh(a/2) * np.tanh(b/2)
    t = np.clip(t, -1 + 1e-10, 1 - 1e-10)
    return 2 * np.arctanh(t)

def g_func(a, b, u_hat):
    """Actualización de LLR para el nodo '+' (cancelación sucesiva) — Ec. (10)"""
    return b + (1 - 2 * u_hat) * a


def sc_decode_n4(llr, frozen_bits):
    """
    Decodificador SC explícito para Polar N=4.
    Convenio: G = kron(F^T, F^T), triangular superior (consistente con §4.2).

    llr:         vector de 4 LLRs del canal (L = 2y para AWGN con σ²=1)
    frozen_bits: dict {index: value} para bits frozen (siempre 0)
    Devuelve:    u_hat (4 bits estimados), x_hat (codeword recuperada)
    """
    L = llr.copy()
    u_hat = np.zeros(4, dtype=int)

    # ── Etapa 1: LLRs intermedios — deshacer Stage 2 del butterfly ────────
    # El encoder combinó (x0,x2) y (x1,x3); f deshace esa mezcla.
    ell_02 = f_func(L[0], L[2])   # ℓ₀₂
    ell_13 = f_func(L[1], L[3])   # ℓ₁₃

    # ── Etapa 2: decodificar u₀ y u₁ ─────────────────────────────────────
    L_u0 = f_func(ell_02, ell_13)
    u_hat[0] = frozen_bits.get(0, 1 if L_u0 < 0 else 0)

    L_u1 = g_func(ell_02, ell_13, u_hat[0])
    u_hat[1] = frozen_bits.get(1, 1 if L_u1 < 0 else 0)

    # ── Etapa 3: g deshace Stage 1 del butterfly con û₀,û₁ conocidos ─────
    beta0 = (u_hat[0] + u_hat[1]) % 2   # = û₀ XOR û₁
    beta1 = u_hat[1]
    g_02 = g_func(L[0], L[2], beta0)
    g_13 = g_func(L[1], L[3], beta1)

    # ── Etapa 4: decodificar u₂ y u₃ ─────────────────────────────────────
    L_u2 = f_func(g_02, g_13)
    u_hat[2] = frozen_bits.get(2, 1 if L_u2 < 0 else 0)

    L_u3 = g_func(g_02, g_13, u_hat[2])
    u_hat[3] = frozen_bits.get(3, 1 if L_u3 < 0 else 0)

    # ── Recuperar x = G·u mod 2 (G triangular superior, convenio §4.2) ────
    G = np.array([[1,1,1,1],[0,1,0,1],[0,0,1,1],[0,0,0,1]], dtype=int)
    x_hat = G @ u_hat % 2

    return u_hat, x_hat, (ell_02, ell_13, L_u0, L_u1, g_02, g_13, L_u2, L_u3)


# ── Verificación determinista: ejemplo numérico §4.2 ──────────────────────
# Transmisión: u=[0,0,1,0] → x=[1,0,1,0] → BPSK: [-1,+1,-1,+1]
# Canal AWGN: y = [-0.8, +1.2, -1.4, +0.6]   (σ²=1 → L=2y)

G4 = np.array([[1,1,1,1],[0,1,0,1],[0,0,1,1],[0,0,0,1]], dtype=int)
frozen = {0: 0, 1: 0}
u_ex   = np.array([0, 0, 1, 0])
x_ex   = G4 @ u_ex % 2
y_ex   = np.array([-0.8, 1.2, -1.4, 0.6])
L_ex   = 2 * y_ex                         # LLR = 2y (σ²=1)

_, _, stages = sc_decode_n4(L_ex, frozen)
ell_02, ell_13, L_u0, L_u1, g_02, g_13, L_u2, L_u3 = stages
u_hat_ex, x_hat_ex, _ = sc_decode_n4(L_ex, frozen)

print('=== Verificación contra ejemplo §4.2 ===')
print(f'u = {u_ex}  →  x = {x_ex}  (esperado [1,0,1,0])')
print(f'y = {y_ex}  →  L_ch = {L_ex}')
print()
print('Etapa 1 — ℓ₀₂ = f(L₀,L₂),  ℓ₁₃ = f(L₁,L₃):')
print(f'  ℓ₀₂ = f({L_ex[0]:.2f}, {L_ex[2]:.2f}) = {ell_02:+.4f}   (esperado ≈ +1.6)')
print(f'  ℓ₁₃ = f({L_ex[1]:.2f}, {L_ex[3]:.2f}) = {ell_13:+.4f}   (esperado ≈ +1.2)')
print()
print('Etapa 2 — decodificar u₀, u₁:')
print(f'  LLR(u₀) = f(ℓ₀₂, ℓ₁₃) = {L_u0:+.4f}  →  û₀ = {u_hat_ex[0]} (frozen)   (esperado ≈ +1.2 → 0)')
print(f'  LLR(u₁) = g(ℓ₀₂, ℓ₁₃, 0) = {L_u1:+.4f}  →  û₁ = {u_hat_ex[1]} (frozen)  (esperado ≈ +2.8 → 0)')
print()
print('Etapa 3 — g₀₂ = g(L₀,L₂,β₀),  g₁₃ = g(L₁,L₃,β₁):')
print(f'  β₀ = û₀ ⊕ û₁ = {(u_hat_ex[0]+u_hat_ex[1])%2}')
print(f'  g₀₂ = g({L_ex[0]:.2f}, {L_ex[2]:.2f}, {(u_hat_ex[0]+u_hat_ex[1])%2}) = {g_02:+.4f}   (esperado = -4.4)')
print(f'  g₁₃ = g({L_ex[1]:.2f}, {L_ex[3]:.2f}, {u_hat_ex[1]}) = {g_13:+.4f}   (esperado = +3.6)')
print()
print('Etapa 4 — decodificar u₂, u₃:')
print(f'  LLR(u₂) = f(g₀₂, g₁₃) = {L_u2:+.4f}  →  û₂ = {u_hat_ex[2]}   (esperado ≈ -3.6 → 1)')
print(f'  LLR(u₃) = g(g₀₂, g₁₃, 1) = {L_u3:+.4f}  →  û₃ = {u_hat_ex[3]}   (esperado = +8.0 → 0)')
print()
print(f'u_hat = {u_hat_ex}   x_hat = {x_hat_ex}')
print(f'Recuperación: {"CORRECTA ✓" if np.array_equal(x_hat_ex, x_ex) else "INCORRECTA ✗"}')

# ── Prueba adicional: canal AWGN aleatorio a 5 dB (100 bloques) ───────────
print('\n=== Prueba MC: SNR=5 dB, 100 bloques ===')
rng2 = np.random.default_rng(7)
SNR_dB = 5.0
sigma2 = 1 / (2 * 10**(SNR_dB/10))   # varianza del ruido (N₀/2)
sigma  = np.sqrt(sigma2)
n_blocks_test, n_correct = 100, 0
for _ in range(n_blocks_test):
    u_rand = np.array([0, 0, rng2.integers(0,2), rng2.integers(0,2)])
    x_rand = G4 @ u_rand % 2
    bpsk   = 1 - 2 * x_rand
    y_rand = bpsk + sigma * rng2.standard_normal(4)
    llr_r  = 2 * y_rand / sigma2
    u_hat_r, x_hat_r, _ = sc_decode_n4(llr_r, frozen)
    if np.array_equal(x_hat_r, x_rand):
        n_correct += 1
ber_approx = 1 - n_correct / n_blocks_test
print(f'Bloques correctos: {n_correct}/100  (BER_bloque ≈ {ber_approx:.2f} @ {SNR_dB} dB)')

In [ ]:
# -- FIG: sc-decoding-n4.png -- Visualizacion del decodificador SC N=4 ------
# Dos pasadas del ejemplo numerico de §4.2
# N=4, k=2, frozen={u0,u1}, LLRs=[-1.6, +2.4, -2.8, +1.2] (min-sum)

import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

# Valores del ejemplo (min-sum, coherentes con el texto)
L_ch = np.array([-1.6, 2.4, -2.8, 1.2])
# Pass 1: f(L0,L2)=+1.6, f(L1,L3)=+1.2
# Decisiones u0: f(+1.6,+1.2)=+1.2; u1: g(+1.6,+1.2,0)=+2.8
# Pass 2: g(L0,L2,0)=-4.4, g(L1,L3,0)=+3.6
# Decisiones u2: f(-4.4,+3.6)=-3.6; u3: g(-4.4,+3.6,1)=+8.0

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
fig.suptitle(
    u'Decodificación SC — N=4, k=2  |  LLRs de canal: [-1.6, +2.4, -2.8, +1.2]',
    fontsize=11, fontweight='bold', y=1.01
)

C_WIRE  = '#CCCCCC'
C_P1    = '#D4813A'
C_P2    = '#3A9E72'
C_FROZ  = '#C96B55'
C_INFO  = '#3672AD'
C_CHAN  = '#666666'
C_KNOWN = '#AAAAAA'

xL, xM, xR = 0.16, 0.50, 0.83
YR = [0.88, 0.62, 0.38, 0.12]
YM = [0.75, 0.25]

def node(ax, x, y, txt, color, fs=8.5, bw=0.17, bh=0.10):
    box = FancyBboxPatch((x-bw/2, y-bh/2), bw, bh,
                         boxstyle='round,pad=0.02', lw=1.4,
                         edgecolor=color, facecolor='white', zorder=4)
    ax.add_patch(box)
    ax.text(x, y, txt, ha='center', va='center',
            fontsize=fs, color=color, fontweight='bold', zorder=5)

def arrow(ax, x1, y1, x2, y2, color, lw=1.8):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=lw), zorder=3)

def wire(ax, x1, y1, x2, y2):
    ax.plot([x1,x2],[y1,y2], color=C_WIRE, lw=1.2, zorder=1)

def setup(ax, title, subtitle=''):
    ax.set_xlim(0,1); ax.set_ylim(0,1); ax.axis('off')
    ax.set_title(f'{title}\n{subtitle}', fontsize=9.5, fontweight='bold', pad=6)
    for x, lbl in [(xL,'Decisiones'),(xM,'Intermedios'),(xR,'Canal LLR')]:
        ax.text(x, 0.98, lbl, ha='center', va='top', fontsize=8, color='#777777', style='italic')
    for x in [0.33, 0.67]:
        ax.axvline(x, color='#F0F0F0', lw=0.8, zorder=0)

# ── PASS 1 ──────────────────────────────────────────────────────────────────
ax = axes[0]
setup(ax, u'Pasada 1 — û₀ y û₁',
      u'operación f  ← right-to-left')
for i, (y, val) in enumerate(zip(YR, L_ch)):
    node(ax, xR, y, f'L{i}={val:+.1f}', C_CHAN, fs=8)
arrow(ax, xR, YR[0], xM, YM[0], C_P1)
arrow(ax, xR, YR[2], xM, YM[0], C_P1)
arrow(ax, xR, YR[1], xM, YM[1], C_P1)
arrow(ax, xR, YR[3], xM, YM[1], C_P1)
node(ax, xM, YM[0], u'f(L₀,L₂)\n= +1.6', C_P1, fs=8)
node(ax, xM, YM[1], u'f(L₁,L₃)\n= +1.2', C_P1, fs=8)
ax.text(0.29, (YM[0]+YM[1])/2+0.04, 'f(A,B)', ha='center', va='bottom',
        fontsize=7.5, color=C_P1, style='italic')
ax.text(0.29, (YM[0]+YM[1])/2-0.04, 'g(A,B,0)', ha='center', va='top',
        fontsize=7.5, color=C_P1, style='italic')
arrow(ax, xM, YM[0], xL, YR[0], C_P1)
arrow(ax, xM, YM[1], xL, YR[0], C_P1)
arrow(ax, xM, YM[0], xL, YR[1], C_P1)
arrow(ax, xM, YM[1], xL, YR[1], C_P1)
node(ax, xL, YR[0], u'û₀ = 0\nLLR=+1.2', C_FROZ, fs=8)
node(ax, xL, YR[1], u'û₁ = 0\nLLR=+2.8', C_FROZ, fs=8)
node(ax, xL, YR[2], u'û₂ = ?', '#BBBBBB', fs=8)
node(ax, xL, YR[3], u'û₃ = ?', '#BBBBBB', fs=8)

# ── PASS 2 ──────────────────────────────────────────────────────────────────
ax = axes[1]
setup(ax, u'Pasada 2 — û₂ y û₃',
      u'operación g  ← usando û₀=0, û₁=0 conocidos')
for i, (y, val) in enumerate(zip(YR, L_ch)):
    node(ax, xR, y, f'L{i}={val:+.1f}', C_CHAN, fs=8)
arrow(ax, xR, YR[0], xM, YM[0], C_P2)
arrow(ax, xR, YR[2], xM, YM[0], C_P2)
arrow(ax, xR, YR[1], xM, YM[1], C_P2)
arrow(ax, xR, YR[3], xM, YM[1], C_P2)
ax.text(0.63, YM[0]+0.12, u'g(·,·, û₀⊕û₁=0)', ha='right',
        va='center', fontsize=7, color=C_P2, style='italic')
ax.text(0.63, YM[1]+0.10, u'g(·,·, û₁=0)', ha='right',
        va='center', fontsize=7, color=C_P2, style='italic')
node(ax, xM, YM[0], u'g(L₀,L₂,0)\n= −4.4', C_P2, fs=8)
node(ax, xM, YM[1], u'g(L₁,L₃,0)\n= +3.6', C_P2, fs=8)
ax.text(0.29, (YM[0]+YM[1])/2+0.04, 'f(A,B)', ha='center', va='bottom',
        fontsize=7.5, color=C_P2, style='italic')
ax.text(0.29, (YM[0]+YM[1])/2-0.04, u'g(A,B,û₂)', ha='center', va='top',
        fontsize=7.5, color=C_P2, style='italic')
arrow(ax, xM, YM[0], xL, YR[2], C_P2)
arrow(ax, xM, YM[1], xL, YR[2], C_P2)
arrow(ax, xM, YM[0], xL, YR[3], C_P2)
arrow(ax, xM, YM[1], xL, YR[3], C_P2)
node(ax, xL, YR[0], u'û₀ = 0\n(congelado)', C_KNOWN, fs=7.5)
node(ax, xL, YR[1], u'û₁ = 0\n(congelado)', C_KNOWN, fs=7.5)
node(ax, xL, YR[2], u'û₂ = 1  ✓\nLLR=−3.6', C_INFO, fs=8)
node(ax, xL, YR[3], u'û₃ = 0  ✓\nLLR=+8.0', C_INFO, fs=8)

legend_els = [
    mpatches.Patch(fc='white', ec=C_P1, lw=1.5, label=u'Pasada 1: f(a,b) = 2arctanh(tanh(a/2)·tanh(b/2))'),
    mpatches.Patch(fc='white', ec=C_P2, lw=1.5, label=u'Pasada 2: g(a,b,û) = b + (1−2û)·a'),
    mpatches.Patch(fc='white', ec=C_FROZ, lw=1.5, label=u'Bit congelado (= 0, conocido)'),
    mpatches.Patch(fc='white', ec=C_INFO, lw=1.5, label=u'Bit de información (decidido)'),
]
fig.legend(handles=legend_els, loc='lower center', ncol=2, fontsize=8,
           framealpha=0.9, edgecolor='#CCCCCC', bbox_to_anchor=(0.5, -0.06))

plt.tight_layout()
plt.savefig('figures/sc-decoding-n4.png', dpi=150, bbox_inches='tight')
plt.show()
print('FIG sc-decoding-n4.png guardada.')


In [ ]:
# -- Ejercicio 4 (cont.) -- Polar N=64: encoder y bits congelados ----
# Encoder: G_64 = F_T^(kron n), F_T=[[1,1],[0,1]] triangular superior (coherente con f/g decoder y bhattacharyya_tree)
# bhattacharyya_tree definida en celda 12 -- no redefinir aqui

def build_polar_G(N):
    """G_N = F_T^(kron n), F_T = [[1,1],[0,1]], sin bit-reversal.
    N debe ser potencia de 2.
    """
    F = np.array([[1, 1], [0, 1]], dtype=int)
    n = int(np.log2(N))
    G = F.copy()
    for _ in range(n - 1):
        G = np.kron(G, F)   # crece: 2->4->8->16->32->64
    return G

def polar_encode(u, G):
    """Codifica u (vector N bits) -> codeword x = G*u mod 2."""
    return G @ u % 2

# Verificacion: G_4 de referencia [[1,0,0,0],[1,1,0,0],[1,0,1,0],[1,1,1,1]]
G4_ref = np.array([[1,1,1,1],[0,1,0,1],[0,0,1,1],[0,0,0,1]], dtype=int)
assert np.array_equal(build_polar_G(4), G4_ref), "ERROR: G_4 no coincide con referencia"
print("build_polar_G(4) verificado contra referencia")

G64 = build_polar_G(64)
N_polar, k_polar = 64, 32
Rc_polar = k_polar / N_polar

# Bits congelados: parametro de Bhattacharyya AWGN, punto de diseno 3 dB
# Z_0 = exp(-Rc * EbN0_lin): aproximacion AWGN (Arikan 2009)
EbN0_design_lin = 10**(3.0 / 10)
Z0_design  = np.exp(-Rc_polar * EbN0_design_lin)
Z_polar    = bhattacharyya_tree(Z0_design, N_polar)
info_idx   = list(np.argsort(Z_polar)[:k_polar])
frozen_set = set(np.argsort(Z_polar)[k_polar:])

print(f"Z_0 = {Z0_design:.4f}  (Rc={Rc_polar}, Eb/N0=3 dB)")
print(f"Canales info: {len(info_idx)},  Canales congelados: {len(frozen_set)}")
print(f"Canales con Z < 0.05 (casi perfectos): {np.sum(Z_polar < 0.05)}")
print(f"Canales con Z > 0.95 (casi inutiles):  {np.sum(Z_polar > 0.95)}")

# -- FIG-06: polar-butterfly.png -- Red butterfly N=8 ----------------------

def draw_butterfly(N, frozen_set, ax):
    """Dibuja la red butterfly de Arikan para codigo Polar de longitud N."""
    n      = int(np.log2(N))
    node_x = np.linspace(0.1, 0.9, n + 2)
    node_y = np.linspace(0.05, 0.95, N)[::-1]

    for s in range(n):
        stride = 2 ** (s + 1)
        half   = stride // 2
        x_in   = node_x[s + 1]
        x_out  = node_x[s + 2]
        for base in range(0, N, stride):
            for offset in range(half):
                i_top = base + offset
                i_bot = base + offset + half
                xmid  = (x_in + x_out) / 2
                ax.plot([x_in, x_out], [node_y[i_top], node_y[i_top]],
                        'k-', lw=1.0, alpha=0.55, zorder=1)
                ax.plot([x_in, xmid], [node_y[i_bot], node_y[i_top]],
                        'k-', lw=1.0, alpha=0.55, zorder=1)
                ax.plot([xmid, x_out], [node_y[i_top], node_y[i_top]],
                        'k-', lw=1.0, alpha=0.55, zorder=1)
                ax.plot([xmid, x_out], [node_y[i_bot], node_y[i_bot]],
                        'k-', lw=1.0, alpha=0.55, zorder=1)
                ax.plot(xmid, node_y[i_top], 'o', color='white', ms=9,
                        markeredgecolor='k', markeredgewidth=0.8, zorder=3)
                ax.text(xmid, node_y[i_top], '+',
                        ha='center', va='center', fontsize=7, fontweight='bold', zorder=4)

    for i in range(N):
        color = 'salmon' if i in frozen_set else 'steelblue'
        ax.plot(node_x[0], node_y[i], 'o', color=color, ms=14, zorder=5,
                markeredgecolor='k', markeredgewidth=0.5)
        lbl = f'$u_{{{i}}}$' + (' (f)' if i in frozen_set else '')
        ax.text(node_x[0] - 0.04, node_y[i], lbl,
                ha='right', va='center', fontsize=7.5)

    for i in range(N):
        ax.plot(node_x[-1], node_y[i], 's', color='lightgray', ms=10, zorder=5,
                markeredgecolor='k', markeredgewidth=0.5)
        ax.text(node_x[-1] + 0.03, node_y[i], f'$x_{{{i}}}$',
                ha='left', va='center', fontsize=7.5)

    ax.axis('off')

N_fig      = 8
Z_fig      = bhattacharyya_tree(np.exp(-0.5 * 2.0), N_fig)
frozen_fig = set(np.argsort(Z_fig)[N_fig // 2:])

fig, ax = plt.subplots(figsize=(10, 5))
draw_butterfly(N_fig, frozen_fig, ax)
ax.set_title('Red butterfly Arikan -- codigo Polar (N=8, k=4, tasa 1/2)\n'
             'Azul = bits de informacion  |  Salmon = bits congelados (= 0)',
             fontsize=10)
plt.tight_layout()
plt.savefig('figures/polar-butterfly.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# -- FIG: polar-butterfly-n4.png -- Red butterfly N=4 con valores del ejemplo --
# N=4, k=2, tasa 1/2. frozen_set={0,1}, u=[0,0,1,0] -> x=[1,0,1,0]
# Reutiliza draw_butterfly (definida en celda 15) y bhattacharyya_tree

import matplotlib.patches as mpatches

N4       = 4
frozen4  = {0, 1}          # bits congelados: u0, u1
u_ex     = [0, 0, 1, 0]    # vector de entrada del ejemplo
w_ex     = [0, 0, 1, 0]    # valores intermedios etapa 1
x_ex     = [1, 0, 1, 0]    # codeword de salida

fig, ax = plt.subplots(figsize=(8, 5))

# Dibujar la red butterfly base con draw_butterfly
draw_butterfly(N4, frozen4, ax)

# Sobreescribir nodos de salida con color por valor de x
n_stages4 = int(np.log2(N4))
node_x4   = np.linspace(0.1, 0.9, n_stages4 + 2)
node_y4   = np.linspace(0.05, 0.95, N4)[::-1]

for i in range(N4):
    color_out = 'mediumseagreen' if x_ex[i] == 0 else 'tomato'
    ax.plot(node_x4[-1], node_y4[i], 's', color=color_out, ms=13, zorder=6,
            markeredgecolor='k', markeredgewidth=0.8)
    ax.text(node_x4[-1] + 0.03, node_y4[i], f'$x_{i}={x_ex[i]}$',
            ha='left', va='center', fontsize=8.5, zorder=7)

# Anotar valores de entrada u_i
for i in range(N4):
    lbl = f'$u_{i}={u_ex[i]}$' + (' (f)' if i in frozen4 else '')
    ax.text(node_x4[0] - 0.04, node_y4[i], lbl,
            ha='right', va='center', fontsize=8.5, zorder=7)

# Anotar valores intermedios w_i entre las dos etapas
x_mid4 = (node_x4[1] + node_x4[2]) / 2
for i in range(N4):
    ax.text(x_mid4, node_y4[i] + 0.04, f'$w_{i}={w_ex[i]}$',
            ha='center', va='bottom', fontsize=7.5, color='dimgray', zorder=7)

# Leyenda
legend_elements = [
    mpatches.Patch(facecolor='salmon',         label='Bit congelado (= 0)'),
    mpatches.Patch(facecolor='steelblue',      label='Bit de informacion'),
    mpatches.Patch(facecolor='mediumseagreen', label='Salida x = 0'),
    mpatches.Patch(facecolor='tomato',         label='Salida x = 1'),
]
ax.legend(handles=legend_elements, fontsize=8, loc='lower right')

ax.set_title('Red butterfly Arikan -- codigo Polar N=4, k=2, tasa 1/2\n'
             'Ejemplo: u=[0,0,1,0]  ->  w=[0,0,1,0]  ->  x=[1,0,1,0]',
             fontsize=10)
plt.tight_layout()
plt.savefig('figures/polar-butterfly-n4.png', dpi=150, bbox_inches='tight')
plt.show()
print('FIG polar-butterfly-n4.png guardada.')


In [ ]:
# -- SC recursivo general y SCL-L=8 --
# Requiere: G_N con F_T=[[1,1],[0,1]] triangular superior (celda 15)
#           f_func / g_func definidos en celda 14
#
# Clave: g_func debe usar la suma parcial x_L = G_{N/2} @ u_hat_izq (no u_hat[i]).
# La red butterfly aplica G_{N/2} a los bits izquierdos antes de hacer XOR,
# por lo que el decoder debe cancelar esa codificacion, no los bits crudos.

def _make_G_cache(N):
    """Precalcula G_k para k = 1, 2, 4, ..., N (necesarias para sumas parciales)."""
    cache = {1: np.array([[1]], dtype=int)}
    k = 2
    while k <= N:
        cache[k] = build_polar_G(k)
        k *= 2
    return cache

def _llr_for_bit(llr_ch, u_hat, target, N, G):
    """LLR para el bit target dado el historial u_hat[0..target-1].
    Rama izquierda: f_func (combina canal).
    Rama derecha: g_func con suma parcial x_L = G_{half} @ u_hat_izq.
    """
    def recurse(llr, start, length):
        if length == 1:
            return llr[0]
        half = length // 2
        if target < start + half:
            llr_l = np.array([f_func(llr[i], llr[i + half]) for i in range(half)])
            return recurse(llr_l, start, half)
        else:
            u_l = u_hat[start:start + half]
            x_L = G[half] @ u_l % 2        # suma parcial: codificar bits izquierdos
            llr_r = np.array([g_func(llr[i], llr[i + half], x_L[i]) for i in range(half)])
            return recurse(llr_r, start + half, half)
    return recurse(llr_ch, 0, N)


def sc_decode_polar(llr_ch, frozen_set, N):
    """Decodificador SC general para codigo Polar de longitud N."""
    G = _make_G_cache(N)
    u_hat = np.zeros(N, dtype=int)
    for i in range(N):
        llr_i = _llr_for_bit(llr_ch, u_hat, i, N, G)
        u_hat[i] = 0 if (i in frozen_set or llr_i >= 0) else 1
    return u_hat


def scl_decode_polar(llr_ch, frozen_set, N, L=8):
    """SCL decoder: beam search con lista de tamano L."""
    G = _make_G_cache(N)
    paths = [(0.0, np.zeros(N, dtype=int))]
    for i in range(N):
        new_paths = []
        for pm, u_hat in paths:
            llr_i = _llr_for_bit(llr_ch, u_hat, i, N, G)
            candidates = [0] if i in frozen_set else [0, 1]
            for bit in candidates:
                u_new = u_hat.copy()
                u_new[i] = bit
                pm_delta = np.log1p(np.exp(-(1 - 2*bit) * llr_i))
                new_paths.append((pm + pm_delta, u_new))
        new_paths.sort(key=lambda x: x[0])
        paths = new_paths[:L]
    return paths[0][1]


# ── Verificacion: SC general debe reproducir sc_decode_n4 (celda 14) ──────
G4_up = np.array([[1,1,1,1],[0,1,0,1],[0,0,1,1],[0,0,0,1]], dtype=int)
frozen4 = {0, 1}
y_ver   = np.array([-0.8, 1.2, -1.4, 0.6])
L_ver   = 2 * y_ver

u_sc    = sc_decode_polar(L_ver, frozen4, 4)
u_scl   = scl_decode_polar(L_ver, frozen4, 4, L=8)
x_sc    = G4_up @ u_sc  % 2
x_scl   = G4_up @ u_scl % 2

print("=== Verificacion con ejemplo seccion 4.2 ===")
print(f"SC  u_hat={u_sc}  x_hat={x_sc}   (esperado [0,0,1,0] / [1,0,1,0])")
print(f"SCL u_hat={u_scl}  x_hat={x_scl}   (esperado [0,0,1,0] / [1,0,1,0])")
assert np.array_equal(u_sc,  [0,0,1,0]), "ERROR: SC fallo"
assert np.array_equal(u_scl, [0,0,1,0]), "ERROR: SCL fallo"
print("OK: ambos decoders recuperan u=[0,0,1,0]")

# ── MC BER: SC vs SCL, N=64, k=32, SNR 0..5 dB ───────────────────────────
# Reutiliza G64, frozen_set, info_idx, Rc_polar, N_polar, k_polar de celda 15
print("\n=== MC BER: SC vs SCL, N=64, k=32 (n_blocks=80 por punto) ===")
rng_sc = np.random.default_rng(42)
snr_range = np.arange(0, 5.5, 1.0)
ber_sc_list, ber_scl_list = [], []
n_blocks_mc = 80

for snr_dB in snr_range:
    sigma2 = 1 / (2 * Rc_polar * 10**(snr_dB/10))
    sigma  = np.sqrt(sigma2)
    err_sc = err_scl = 0
    for _ in range(n_blocks_mc):
        u_tx = np.zeros(N_polar, dtype=int)
        u_tx[info_idx] = rng_sc.integers(0, 2, size=k_polar)
        x_tx = G64 @ u_tx % 2
        bpsk = 1 - 2*x_tx
        y    = bpsk + sigma * rng_sc.standard_normal(N_polar)
        llr  = 2 * y / sigma2
        u_sc_mc  = sc_decode_polar(llr, frozen_set, N_polar)
        u_scl_mc = scl_decode_polar(llr, frozen_set, N_polar, L=8)
        err_sc  += np.sum(u_sc_mc[info_idx]  != u_tx[info_idx])
        err_scl += np.sum(u_scl_mc[info_idx] != u_tx[info_idx])
    ber_sc_list.append(err_sc  / (n_blocks_mc * k_polar))
    ber_scl_list.append(err_scl / (n_blocks_mc * k_polar))
    print(f"  Eb/N0={snr_dB:.0f} dB — SC BER={ber_sc_list[-1]:.3f}  SCL BER={ber_scl_list[-1]:.3f}")

print("MC BER completado.")


In [12]:
# -- polar-polarization.png (FIG-07) + MC BER: SC vs SCL-8 ---------------
# NOTA: la figura de polarizacion se genera primero; el MC BER es opcional

# ── FIG-07: polar-polarization.png ───────────────────────────────────────
Z_frozen_arr = Z_polar[list(frozen_set)]
Z_info_arr   = Z_polar[list(info_idx)]

assert np.sum(Z_polar < 0.1) >= 10 and np.sum(Z_polar > 0.9) >= 10, \
    'Polarizacion debil: revisar Z_0'

bins = np.linspace(0, 1, 31)
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(Z_frozen_arr, bins=bins, color='salmon',    alpha=0.7,
        label=f'Bits congelados ({len(frozen_set)})', edgecolor='white', lw=0.3)
ax.hist(Z_info_arr,   bins=bins, color='steelblue', alpha=0.7,
        label=f'Bits de informacion ({len(info_idx)})', edgecolor='white', lw=0.3)
ax.axvline(0.5, color='gray', ls='--', lw=1.2, label='Umbral de seleccion')
ax.set_xlabel('Parametro de Bhattacharyya $Z(W_{64}^{(i)})$')
ax.set_ylabel('Numero de canales sinteticos')
ax.set_title('Polarizacion del canal -- $N=64$, diseno $E_b/N_0=3$ dB, $r_c=1/2$\n'
             'Canales sinteticos polarizados hacia $Z\\approx 0$ (buenos) y $Z\\approx 1$ (malos)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('figures/polar-polarization.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'FIG-07 guardada. Z<0.05: {np.sum(Z_polar<0.05)}, Z>0.95: {np.sum(Z_polar>0.95)}')
print('SC/SCL N=64 (comparativa BER) diferido — ver Fase 4 / LAB-02')


FIG-07 guardada. Z<0.05: 27, Z>0.95: 12
SC/SCL N=64 (comparativa BER) diferido — ver Fase 4 / LAB-02


/tmp/claude-1000/ipykernel_867002/1314241132.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [13]:
# -- waterfall-curves.png (FIG-03 / LAB-05): comparativa LDPC vs Polar vs BPSK sin codigo --
# Reutiliza H12, var_nbrs12, chk_nbrs12, Rc12, bp_awgn de celda 8
# Polar: cota de union P_b <= sum(Z_polar[info_idx]) / k  (referencia teorica, no simulacion SC)

rng_wf = np.random.default_rng(2024)

EbN0_wf = np.arange(0.0, 5.5, 0.5)       # puntos MC LDPC
EbN0_plot_wf = np.linspace(-1, 6, 300)   # curva continua BPSK

# ── BPSK sin codigo (analitica) ──────────────────────────────────────────
BER_bpsk_wf = 0.5 * erfc(np.sqrt(10**(EbN0_plot_wf / 10)))

# ── LDPC Monte Carlo (n=240, tasa ~1/2) — reutiliza H12/bp_awgn de celda 8 ─
def run_mc_ber_wf(H, var_nbrs, chk_nbrs, Rc, EbN0_dB_arr, n_blocks=150, max_iter=30):
    n = H.shape[1]
    floor = 1.0 / (n_blocks * n)
    c_test = np.zeros(n, dtype=int)
    bpsk   = np.ones(n)
    BER = []
    for EbN0_dB in EbN0_dB_arr:
        EbN0_lin = 10**(EbN0_dB / 10)
        sigma2 = 1.0 / (2 * Rc * EbN0_lin)
        sigma  = np.sqrt(sigma2)
        bit_errors = 0
        for _ in range(n_blocks):
            y = bpsk + sigma * rng_wf.standard_normal(n)
            c_hat, _ = bp_awgn(H, 2*y/sigma2, var_nbrs, chk_nbrs, max_iter)
            bit_errors += np.sum(c_hat != c_test)
        BER.append(max(bit_errors / (n_blocks * n), floor))
    return np.array(BER)

print('Simulando waterfall LDPC tasa 1/2 (~20 s) ...')
BER_ldpc_wf = run_mc_ber_wf(H12, var_nbrs12, chk_nbrs12, Rc12, EbN0_wf, n_blocks=150)

# ── Cota de union Polar (referencia teorica) ─────────────────────────────
# P_b <= (1/k) * sum_{i in info_idx} Z(W_N^{(i)})  normalizada por EbN0
# Z_polar y info_idx definidos en celda 15
EbN0_polar_wf = np.arange(0.0, 5.5, 0.5)
BER_polar_bound = []
for EbN0_dB in EbN0_polar_wf:
    EbN0_lin = 10**(EbN0_dB / 10)
    Z0_cur = np.exp(-Rc_polar * EbN0_lin)
    Z_cur  = bhattacharyya_tree(Z0_cur, N_polar)
    pb_upper = np.sum(Z_cur[list(info_idx)]) / k_polar
    BER_polar_bound.append(min(pb_upper, 1.0))
BER_polar_bound = np.array(BER_polar_bound)

# ── Figura ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(EbN0_plot_wf, BER_bpsk_wf,     'k-',  lw=2, label='BPSK sin codigo')
ax.semilogy(EbN0_wf,      BER_ldpc_wf,     'o-',  color='steelblue',  lw=2, ms=5,
            label=f'LDPC $r_c\\approx{Rc12:.2f}$ (n=240, Monte Carlo)')
ax.semilogy(EbN0_polar_wf, BER_polar_bound, 's--', color='darkorange', lw=2, ms=5,
            label=f'Polar $r_c=0.5$ (N=64, cota de union Bhattacharyya)')

ax.set_xlabel('$E_b/N_0$ (dB)')
ax.set_ylabel('BER')
ax.set_title('Curvas waterfall — LDPC vs Polar vs BPSK sin codigo\n'
             'Polar: cota teorica (decodificador SC diferido — Fase 4)')
ax.set_ylim(1e-5, 1.1)
ax.set_xlim(-1, 6)
ax.legend(fontsize=9, loc='lower left')
plt.tight_layout()
plt.savefig('figures/waterfall-curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('FIG-03 waterfall-curves.png guardada.')


Simulando waterfall LDPC tasa 1/2 (~20 s) ...


FIG-03 waterfall-curves.png guardada.


/tmp/claude-1000/ipykernel_867002/1031376612.py:65: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Ejercicio 6 — Integrador OFDM+LDPC: canal frequency-selective

En este ejercicio conectamos el transceptor OFDM de la Sesión 03 con el codec LDPC
de esta sesión para construir un sistema de comunicación inalámbrica **end-to-end**
sobre un canal frequency-selective.

El canal de 3 taps $h = [0.8, 0.5, 0.3]$ introduce selectividad en frecuencia que el
ecualizador ZF mitiga a costa de amplificar el ruido en los nulos espectrales.
El objetivo es cuantificar la **ganancia de codificación** que aporta el LDPC (n=240)
sobre este canal real y comparar con la referencia AWGN ideal.

In [14]:
# ── Ejercicio 6 — Integrador OFDM+LDPC: canal frequency-selective ──────────
# Reutilización directa de las 4 funciones OFDM de la sesión 03 (sin modificación)

# ── a) Funciones OFDM de sesión 03 ─────────────────────────────────────────
def ofdm_tx(X, N_CP):
    """IFFT normalizada + prefijo cíclico."""
    x = np.fft.ifft(X, norm='ortho')
    return np.concatenate([x[-N_CP:], x])

def apply_channel(x_signal, h):
    """Convolución lineal del canal multipath."""
    return np.convolve(x_signal, h, mode='full')[:len(x_signal)]

def ofdm_rx_no_channel(x_with_cp, N, N_CP):
    """Eliminar CP + FFT normalizada."""
    return np.fft.fft(x_with_cp[N_CP:], norm='ortho')

def zf_equalizer(Y, h, N):
    """ZF: Y[k] / H[k] — elimina distorsión, amplifica ruido en nulos."""
    return Y / np.fft.fft(h, n=N)

# ── b) Parámetros del sistema ───────────────────────────────────────────────
N_ofdm  = 128                            # subportadoras OFDM
N_CP    = 16                             # prefijo cíclico (> L-1 = 2)
h_ch    = np.array([0.8, 0.5, 0.3])     # canal 3-tap frequency-selective
H_k_ch  = np.fft.fft(h_ch, n=N_ofdm)   # respuesta frecuencial del canal
H_k_mag2_full = np.maximum(np.abs(H_k_ch)**2, 1e-9)   # |H[k]|² (clampado)

# LDPC: usa H12, var_nbrs12, chk_nbrs12, Rc12, bp_awgn de Cell 8
# 240 bits → 120 símbolos QPSK → caben en 1 trama OFDM de N_ofdm=128 (8 subc. padding)
n_ldpc_ex6 = H12.shape[1]              # = 240
n_active   = n_ldpc_ex6 // 2           # = 120 subportadoras activas
H_k_mag2   = H_k_mag2_full[:n_active]  # |H[k]|² para k=0..119

# ── c) QPSK helpers ────────────────────────────────────────────────────────
def qpsk_map_ex6(bits):
    """2 bits → símbolo QPSK Gray (|s|²=1): b0=bit_I, b1=bit_Q."""
    b0 = bits[0::2].astype(float)
    b1 = bits[1::2].astype(float)
    return ((1 - 2*b0) + 1j*(1 - 2*b1)) / np.sqrt(2)

def qpsk_hard_ex6(X_eq):
    """Decisión dura QPSK: Re<0 → b_I=1, Im<0 → b_Q=1."""
    b = np.zeros(2*len(X_eq), dtype=int)
    b[0::2] = (X_eq.real < 0).astype(int)
    b[1::2] = (X_eq.imag < 0).astype(int)
    return b

def qpsk_llr_ex6(X_eq, sigma2_noise, H_k_mag2_active):
    """
    LLR por bit para QPSK después de ecualizador ZF sobre canal OFDM.

    Tras ZF, el ruido efectivo en subportadora k es CN(0, sigma2_noise/|H[k]|²).
    Para QPSK Gray (s_I = ±1/√2), la LLR del bit_I es:
        LLR_I[k] = 2√2 · Re(X_eq[k]) · |H[k]|² / sigma2_noise

    Args:
        X_eq             : símbolos ZF-ecualizados, shape (n_active,)
        sigma2_noise     : varianza compleja del ruido antes de ZF (N0)
        H_k_mag2_active  : |H[k]|² para k = 0..n_active-1

    Returns:
        llr : 2*n_active LLRs [I0,Q0,I1,Q1,...]; positivo → bit=0
    """
    # Derivación: LLR_I = 2*(1/√2)*Re(X_eq) / (sigma2_eff/2)
    #             donde sigma2_eff = sigma2_noise/|H[k]|²
    #             → LLR_I = 2√2 * Re(X_eq) * |H[k]|² / sigma2_noise
    scale = 2.0 * np.sqrt(2) * H_k_mag2_active / sigma2_noise
    llr   = np.zeros(2 * len(X_eq))
    llr[0::2] = scale * X_eq.real   # bits I (b0)
    llr[1::2] = scale * X_eq.imag   # bits Q (b1)
    return llr

# ── d) Simulación Monte Carlo ───────────────────────────────────────────────
rng_ex6   = np.random.default_rng(2025)
n_sim_ex6 = 300                           # tramas Monte Carlo por SNR
EbN0_ex6  = np.arange(0, 10.5, 0.5)      # 21 puntos de SNR
c_zeros   = np.zeros(n_ldpc_ex6, dtype=int)   # codeword válida: H12 @ 0 = 0

ber_ofdm_unc = []
ber_ofdm_cod = []

print('Simulando OFDM sin FEC y OFDM+LDPC... (~45 s)')
for EbN0_dB in EbN0_ex6:
    EbN0_lin  = 10**(EbN0_dB / 10)
    # σ²_noise (varianza compleja): E_s=1 QPSK, k=2 bits/sym
    # Uncoded: σ²_unc = 1/(2·EbN0)          [Eb = E_s/k = 1/2]
    # Coded:   σ²_cod = 1/(2·Rc12·EbN0)     [Eb_info = Eb/Rc]
    sigma2_unc = 1.0 / (2.0 * EbN0_lin)
    sigma2_cod = 1.0 / (2.0 * Rc12 * EbN0_lin)
    floor_v    = 1.0 / (n_sim_ex6 * n_ldpc_ex6)

    err_unc = err_cod = 0

    for _ in range(n_sim_ex6):
        # Uncoded OFDM ─────────────────────────────────────────────────
        bits_u  = rng_ex6.integers(0, 2, 2 * N_ofdm)
        X_u     = qpsk_map_ex6(bits_u)
        x_u     = ofdm_tx(X_u, N_CP)
        y_u     = apply_channel(x_u, h_ch)
        n_u     = np.sqrt(sigma2_unc / 2) * (
            rng_ex6.standard_normal(len(y_u)) + 1j * rng_ex6.standard_normal(len(y_u)))
        Y_u     = ofdm_rx_no_channel(y_u + n_u, N_ofdm, N_CP)
        X_eq_u  = zf_equalizer(Y_u, h_ch, N_ofdm)
        err_unc += np.sum(qpsk_hard_ex6(X_eq_u) != bits_u)

        # OFDM + LDPC ──────────────────────────────────────────────────
        X_c = np.zeros(N_ofdm, dtype=complex)
        X_c[:n_active] = qpsk_map_ex6(c_zeros)    # 240 bits en 120 subportadoras
        x_c     = ofdm_tx(X_c, N_CP)
        y_c     = apply_channel(x_c, h_ch)
        n_c     = np.sqrt(sigma2_cod / 2) * (
            rng_ex6.standard_normal(len(y_c)) + 1j * rng_ex6.standard_normal(len(y_c)))
        Y_c     = ofdm_rx_no_channel(y_c + n_c, N_ofdm, N_CP)
        X_eq_c  = zf_equalizer(Y_c, h_ch, N_ofdm)

        llr_c   = qpsk_llr_ex6(X_eq_c[:n_active], sigma2_cod, H_k_mag2)
        c_hat, _ = bp_awgn(H12, llr_c, var_nbrs12, chk_nbrs12, max_iter=20)
        err_cod += np.sum(c_hat != c_zeros)

    ber_ofdm_unc.append(max(err_unc / (n_sim_ex6 * 2 * N_ofdm), floor_v))
    ber_ofdm_cod.append(max(err_cod / (n_sim_ex6 * n_ldpc_ex6), floor_v))

ber_ofdm_unc = np.array(ber_ofdm_unc)
ber_ofdm_cod = np.array(ber_ofdm_cod)

idx_5dB = np.argmin(np.abs(EbN0_ex6 - 5.0))
print(f'@ 5 dB  — OFDM sin FEC: {ber_ofdm_unc[idx_5dB]:.2e}  OFDM+LDPC: {ber_ofdm_cod[idx_5dB]:.2e}')

# ── e) Figura ofdm-ldpc-ber.png (FIG-09) ───────────────────────────────────
EbN0_plot = np.linspace(0, 10, 300)
BER_awgn  = 0.5 * erfc(np.sqrt(10**(EbN0_plot / 10)))

fig, ax = plt.subplots(figsize=(10, 5))
ax.semilogy(EbN0_plot,  BER_awgn,      'k--', lw=1.5, label='QPSK AWGN (referencia teórica)')
ax.semilogy(EbN0_ex6,  ber_ofdm_unc,  'o-',  color='steelblue', lw=2, ms=5,
            label='OFDM sin FEC (ZF, canal 3-tap)')
ax.semilogy(EbN0_ex6,  ber_ofdm_cod,  's-',  color='darkorange', lw=2, ms=5,
            label=f'OFDM + LDPC $r_c\\approx{Rc12:.2f}$ (BP, n=240)')

ebn0_sh = 10 * np.log10((2**Rc12 - 1) / Rc12)
ax.axvline(ebn0_sh, color='darkorange', ls=':', lw=0.9, alpha=0.6)
ax.text(ebn0_sh + 0.1, 3e-1, f'$C(r={Rc12:.2f})$', fontsize=8,
        color='darkorange', rotation=90, va='top')

ax.set_xlabel('$E_b/N_0$ (dB)')
ax.set_ylabel('BER')
ax.set_title('OFDM+LDPC end-to-end — canal frequency-selective 3-tap + ZF + BP (n=240)')
ax.set_xlim(0, 10)
ax.set_ylim(1e-5, 1.1)
ax.legend(fontsize=9, loc='lower left')
plt.tight_layout()
plt.savefig('figures/ofdm-ldpc-ber.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figura guardada: figures/ofdm-ldpc-ber.png')


Simulando OFDM sin FEC y OFDM+LDPC... (~45 s)


@ 5 dB  — OFDM sin FEC: 5.11e-02  OFDM+LDPC: 2.21e-02


Figura guardada: figures/ofdm-ldpc-ber.png


/tmp/claude-1000/ipykernel_867002/2822372125.py:154: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


---
## Resumen

| Ejercicio | Concepto clave | Resultado esperado |
|---|---|---|
| 1 | Región de Shannon, límite $E_b/N_0$ | $-1{,}59$ dB; gap ~11 dB para 64-QAM |
| 2 | Código Hamming (7,4), síndrome | Síndrome identifica columna = posición del error |
| 3 | BP en LDPC (8,4) BSC | Convergencia en <10 iter para $p=0{,}1$ |
| 4 | Árbol de Bhattacharyya, BEC $\varepsilon=0{,}5$ | 4 canales buenos: $\{3,5,6,7\}$ aprox. |
| 5 | Decodificador SC N=4 | Recuperación correcta a 5 dB SNR |
| 6 | OFDM+LDPC end-to-end (canal freq-selective) | Ganancia de codificación LDPC visible en canal multipath |